In [46]:
import os
from utils.spark_session import get_spark_session

from pyspark.sql.functions import (
    col, coalesce, sum, min, max, avg, to_date, dayofmonth, month, year, array_contains, lit
)


In [47]:
def describe_column(df, categoric_column, numeric_column):
    """
    Computes basic descriptive statistics (min, max, mean) for a numeric column,
    grouped by the categoric column.

    """

    stats_df = (
        df
        .groupBy(categoric_column)
        .agg(
            min(numeric_column).alias(f"min_{numeric_column}"),
            avg(numeric_column).alias(f"mean_{numeric_column}"),
            max(numeric_column).alias(f"max_{numeric_column}")
        )
    )
    
    return stats_df

In [48]:
spark = get_spark_session(app_name="01-eda-raw")

In [49]:
offers_df = spark.read.json(os.path.join('..', 'data', 'raw', 'offers.json'))
customers_df = spark.read.json(os.path.join('..', 'data', 'raw', 'profile.json'))
transactions_df = spark.read.json(os.path.join('..', 'data', 'raw', 'transactions.json'))

In [50]:
offers_df.show(5, truncate=False)

+----------------------------+--------------+--------+--------------------------------+---------+-------------+
|channels                    |discount_value|duration|id                              |min_value|offer_type   |
+----------------------------+--------------+--------+--------------------------------+---------+-------------+
|[email, mobile, social]     |10            |7.0     |ae264e3637204a6fb9bb56bc8210ddfd|10       |bogo         |
|[web, email, mobile, social]|10            |5.0     |4d5c57ea9a6940dd891ad53e9dbe8da0|10       |bogo         |
|[web, email, mobile]        |0             |4.0     |3f207df678b143eea3cee63160fa8bed|0        |informational|
|[web, email, mobile]        |5             |7.0     |9b98b8c7a33c4b65b9aebfe6a799e6d9|5        |bogo         |
|[web, email]                |5             |10.0    |0b1e1539f2cc45b7b9fa7c272da2e1d7|20       |discount     |
+----------------------------+--------------+--------+--------------------------------+---------+-------

In [51]:
channel_list = ["web", "email", "mobile", "social"]

for ch in channel_list:
    offers_df = (offers_df.withColumn(f"{ch}", array_contains("channels", lit(ch)).cast("int")))

offers_df = offers_df.drop("channels")

offers_df.show(5, truncate=False)


+--------------+--------+--------------------------------+---------+-------------+---+-----+------+------+
|discount_value|duration|id                              |min_value|offer_type   |web|email|mobile|social|
+--------------+--------+--------------------------------+---------+-------------+---+-----+------+------+
|10            |7.0     |ae264e3637204a6fb9bb56bc8210ddfd|10       |bogo         |0  |1    |1     |1     |
|10            |5.0     |4d5c57ea9a6940dd891ad53e9dbe8da0|10       |bogo         |1  |1    |1     |1     |
|0             |4.0     |3f207df678b143eea3cee63160fa8bed|0        |informational|1  |1    |1     |0     |
|5             |7.0     |9b98b8c7a33c4b65b9aebfe6a799e6d9|5        |bogo         |1  |1    |1     |0     |
|5             |10.0    |0b1e1539f2cc45b7b9fa7c272da2e1d7|20       |discount     |1  |1    |0     |0     |
+--------------+--------+--------------------------------+---------+-------------+---+-----+------+------+
only showing top 5 rows



In [52]:
customers_df.show(5, truncate=False)

+---+-----------------+------+--------------------------------+-------------+
|age|credit_card_limit|gender|id                              |registered_on|
+---+-----------------+------+--------------------------------+-------------+
|118|NULL             |NULL  |68be06ca386d4c31939f3a4f0e3dd783|20170212     |
|55 |112000.0         |F     |0610b486422d4921ae7d2bf64640c50b|20170715     |
|118|NULL             |NULL  |38fe809add3b4fcf9315a9694bb96ff5|20180712     |
|75 |100000.0         |F     |78afa995795e4d85b5d9ceeca43f5fef|20170509     |
|118|NULL             |NULL  |a03223e636434f42ac4c3df47e8bac43|20170804     |
+---+-----------------+------+--------------------------------+-------------+
only showing top 5 rows



In [53]:
transactions_df.show(5, truncate=False)

+--------------------------------+--------------+---------------------+----------------------------------------------------+
|account_id                      |event         |time_since_test_start|value                                               |
+--------------------------------+--------------+---------------------+----------------------------------------------------+
|78afa995795e4d85b5d9ceeca43f5fef|offer received|0.0                  |{NULL, 9b98b8c7a33c4b65b9aebfe6a799e6d9, NULL, NULL}|
|a03223e636434f42ac4c3df47e8bac43|offer received|0.0                  |{NULL, 0b1e1539f2cc45b7b9fa7c272da2e1d7, NULL, NULL}|
|e2127556f4f64592b11af22de27a7932|offer received|0.0                  |{NULL, 2906b810c7d4411798c6938adc9daaa5, NULL, NULL}|
|8ec6ce2a7e7949b1bf142def7d0e0586|offer received|0.0                  |{NULL, fafdcd668e3743c1bb461111dcafc2a4, NULL, NULL}|
|68617ca6246f4fbc85e91a2a49552598|offer received|0.0                  |{NULL, 4d5c57ea9a6940dd891ad53e9dbe8da0, NULL, NULL}|


In [ ]:
transactions_df.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- event: string (nullable = true)
 |-- time_since_test_start: double (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- amount: double (nullable = true)
 |    |-- offer id: string (nullable = true)
 |    |-- offer_id: string (nullable = true)
 |    |-- reward: double (nullable = true)



In [55]:
transactions_df = (
    transactions_df
    .withColumn("amount", col("value.amount"))
    .withColumn("offer_id_tmp1", col("value.`offer id`")) 
    .withColumn("offer_id_tmp2", col("value.offer_id"))
    .withColumn("reward", col("value.reward"))
    .withColumn("offer_id", coalesce(col("offer_id_tmp2"), col("offer_id_tmp1")))
    .drop("offer_id_tmp1", "offer_id_tmp2", "value")
)


In [56]:
transactions_df.show(5, truncate=False)

+--------------------------------+--------------+---------------------+------+------+--------------------------------+
|account_id                      |event         |time_since_test_start|amount|reward|offer_id                        |
+--------------------------------+--------------+---------------------+------+------+--------------------------------+
|78afa995795e4d85b5d9ceeca43f5fef|offer received|0.0                  |NULL  |NULL  |9b98b8c7a33c4b65b9aebfe6a799e6d9|
|a03223e636434f42ac4c3df47e8bac43|offer received|0.0                  |NULL  |NULL  |0b1e1539f2cc45b7b9fa7c272da2e1d7|
|e2127556f4f64592b11af22de27a7932|offer received|0.0                  |NULL  |NULL  |2906b810c7d4411798c6938adc9daaa5|
|8ec6ce2a7e7949b1bf142def7d0e0586|offer received|0.0                  |NULL  |NULL  |fafdcd668e3743c1bb461111dcafc2a4|
|68617ca6246f4fbc85e91a2a49552598|offer received|0.0                  |NULL  |NULL  |4d5c57ea9a6940dd891ad53e9dbe8da0|
+--------------------------------+--------------

In [ ]:
display("offers", offers_df.count(), "customers", customers_df.count(), "transactions", transactions_df.count())

'offers'

10

'customers'

17000

'transactions'

306534

In [58]:
offers_df = offers_df.dropDuplicates()
customers_df = customers_df.dropDuplicates()
transactions_df = transactions_df.dropDuplicates()

In [59]:
# ~ 400 duplicate values in transactions_df

display("offers", offers_df.count(), "customers", customers_df.count(), "transactions", transactions_df.count())

'offers'

10

'customers'

17000

'transactions'

306137

### EDA transactions_df

In [60]:
transactions_df.show(5, truncate=False)

+--------------------------------+--------------+---------------------+------+------+--------------------------------+
|account_id                      |event         |time_since_test_start|amount|reward|offer_id                        |
+--------------------------------+--------------+---------------------+------+------+--------------------------------+
|c4863c7985cf408faee930f111475da3|offer received|0.0                  |NULL  |NULL  |2298d6c36e964ae4a3e7e9706d1fb8c2|
|fae5f722dce445c1ae311729464943cf|offer received|0.0                  |NULL  |NULL  |9b98b8c7a33c4b65b9aebfe6a799e6d9|
|352ccf90feeb4c92a54834c7ad2b66bf|offer received|0.0                  |NULL  |NULL  |4d5c57ea9a6940dd891ad53e9dbe8da0|
|3bcc51fdde354eb1949c813dbc905182|offer received|0.0                  |NULL  |NULL  |5a8bc65990b245e5a138643cd4eb9837|
|267e47de94fd46b1afa96dea1c9d3cbf|offer received|0.0                  |NULL  |NULL  |2298d6c36e964ae4a3e7e9706d1fb8c2|
+--------------------------------+--------------

In [61]:
print(f"shape: {transactions_df.count()}, {len(transactions_df.columns)}")

shape: 306137, 6


In [62]:
transactions_df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in transactions_df.columns
]).show()

+----------+-----+---------------------+------+------+--------+
|account_id|event|time_since_test_start|amount|reward|offer_id|
+----------+-----+---------------------+------+------+--------+
|         0|    0|                    0|167184|272955|  138953|
+----------+-----+---------------------+------+------+--------+



In [63]:
for col_name in transactions_df.columns:
    print(f"{col_name}: {transactions_df.select(col_name).distinct().count()} distinct values")

account_id: 17000 distinct values
event: 4 distinct values
time_since_test_start: 120 distinct values
amount: 5104 distinct values
reward: 5 distinct values
offer_id: 11 distinct values


In [64]:
transactions_df.groupBy("event").count().orderBy("count", ascending=False).show()

+---------------+------+
|          event| count|
+---------------+------+
|    transaction|138953|
| offer received| 76277|
|   offer viewed| 57725|
|offer completed| 33182|
+---------------+------+



In [65]:
describe_column(transactions_df, "event", "amount").orderBy("event").show()

+---------------+----------+------------------+----------+
|          event|min_amount|       mean_amount|max_amount|
+---------------+----------+------------------+----------+
|offer completed|      NULL|              NULL|      NULL|
| offer received|      NULL|              NULL|      NULL|
|   offer viewed|      NULL|              NULL|      NULL|
|    transaction|      0.05|12.777356156398204|   1062.28|
+---------------+----------+------------------+----------+



In [66]:
describe_column(transactions_df, "event", "time_since_test_start").orderBy("event").show()

+---------------+-------------------------+--------------------------+-------------------------+
|          event|min_time_since_test_start|mean_time_since_test_start|max_time_since_test_start|
+---------------+-------------------------+--------------------------+-------------------------+
|offer completed|                      0.0|        16.651731360376107|                    29.75|
| offer received|                      0.0|         13.85747997430418|                     24.0|
|   offer viewed|                      0.0|        14.762104807275877|                    29.75|
|    transaction|                      0.0|        15.899347261304182|                    29.75|
+---------------+-------------------------+--------------------------+-------------------------+



In [67]:
describe_column(transactions_df, "event", "reward").orderBy("event").show()

+---------------+----------+-----------------+----------+
|          event|min_reward|      mean_reward|max_reward|
+---------------+----------+-----------------+----------+
|offer completed|       2.0|4.902627930805859|      10.0|
| offer received|      NULL|             NULL|      NULL|
|   offer viewed|      NULL|             NULL|      NULL|
|    transaction|      NULL|             NULL|      NULL|
+---------------+----------+-----------------+----------+



### EDA customer_df

In [68]:
customers_df.show(5, truncate=False)

+---+-----------------+------+--------------------------------+-------------+
|age|credit_card_limit|gender|id                              |registered_on|
+---+-----------------+------+--------------------------------+-------------+
|20 |49000.0          |M     |576e6eed3c6a4ac682ebd35b7ea672f4|20140531     |
|42 |60000.0          |M     |e6d75ffe3371472b80c2ed8c51dbddaa|20160429     |
|70 |70000.0          |M     |1679b7af5a294c6d9ae961232318ad55|20160819     |
|50 |39000.0          |M     |f7662fcebd8049b280cc95b70caa1f23|20160911     |
|63 |58000.0          |F     |bea7a8fa65284ef98edeeee7cb8abd53|20180314     |
+---+-----------------+------+--------------------------------+-------------+
only showing top 5 rows



In [69]:
print(f"shape: {customers_df.count()}, {len(customers_df.columns)}")

shape: 17000, 5


In [70]:
# Check null values

customers_df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in customers_df.columns
]).show()

+---+-----------------+------+---+-------------+
|age|credit_card_limit|gender| id|registered_on|
+---+-----------------+------+---+-------------+
|  0|             2175|  2175|  0|            0|
+---+-----------------+------+---+-------------+



In [71]:
for col_name in customers_df.columns:
    print(f"{col_name}: {customers_df.select(col_name).distinct().count()} distinct values")

age: 85 distinct values
credit_card_limit: 92 distinct values
gender: 4 distinct values
id: 17000 distinct values
registered_on: 1716 distinct values


In [72]:
customers_df.groupBy("gender").count().orderBy("count", ascending=False).show()

+------+-----+
|gender|count|
+------+-----+
|     M| 8484|
|     F| 6129|
|  NULL| 2175|
|     O|  212|
+------+-----+



In [73]:
describe_column(customers_df, "gender", "credit_card_limit").orderBy("gender").show()

+------+---------------------+----------------------+---------------------+
|gender|min_credit_card_limit|mean_credit_card_limit|max_credit_card_limit|
+------+---------------------+----------------------+---------------------+
|  NULL|                 NULL|                  NULL|                 NULL|
|     F|              30000.0|     71306.41213901126|             120000.0|
|     M|              30000.0|     61194.60160301744|             120000.0|
|     O|              30000.0|    63287.735849056604|             100000.0|
+------+---------------------+----------------------+---------------------+



In [74]:
describe_column(customers_df, "gender", "age").orderBy("gender").show()

+------+-------+-----------------+-------+
|gender|min_age|         mean_age|max_age|
+------+-------+-----------------+-------+
|  NULL|    118|            118.0|    118|
|     F|     18|57.54495023658019|    101|
|     M|     18|52.11669024045262|    100|
|     O|     20|54.40094339622642|    100|
+------+-------+-----------------+-------+



In [75]:
customers_df.filter(col("age") > 110).count()

2175

In [76]:
customers_df = (
    customers_df
    .withColumn("registered_on", to_date("registered_on", "yyyyMMdd"))
    .withColumn("reg_day", dayofmonth("registered_on"))
    .withColumn("reg_month", month("registered_on"))
    .withColumn("reg_year", year("registered_on"))
)

customers_df.show(5, truncate=False)

+---+-----------------+------+--------------------------------+-------------+-------+---------+--------+
|age|credit_card_limit|gender|id                              |registered_on|reg_day|reg_month|reg_year|
+---+-----------------+------+--------------------------------+-------------+-------+---------+--------+
|20 |49000.0          |M     |576e6eed3c6a4ac682ebd35b7ea672f4|2014-05-31   |31     |5        |2014    |
|42 |60000.0          |M     |e6d75ffe3371472b80c2ed8c51dbddaa|2016-04-29   |29     |4        |2016    |
|70 |70000.0          |M     |1679b7af5a294c6d9ae961232318ad55|2016-08-19   |19     |8        |2016    |
|50 |39000.0          |M     |f7662fcebd8049b280cc95b70caa1f23|2016-09-11   |11     |9        |2016    |
|63 |58000.0          |F     |bea7a8fa65284ef98edeeee7cb8abd53|2018-03-14   |14     |3        |2018    |
+---+-----------------+------+--------------------------------+-------------+-------+---------+--------+
only showing top 5 rows



In [77]:
customers_df.groupBy("reg_month").count().orderBy("reg_month").show(12)

+---------+-----+
|reg_month|count|
+---------+-----+
|        1| 1525|
|        2| 1202|
|        3| 1329|
|        4| 1315|
|        5| 1307|
|        6| 1265|
|        7| 1359|
|        8| 1610|
|        9| 1515|
|       10| 1568|
|       11| 1449|
|       12| 1556|
+---------+-----+



In [78]:
customers_df.groupBy("reg_year").count().orderBy("reg_year").show()

+--------+-----+
|reg_year|count|
+--------+-----+
|    2013|  286|
|    2014|  691|
|    2015| 1830|
|    2016| 3526|
|    2017| 6469|
|    2018| 4198|
+--------+-----+



### EDA offers_df

In [79]:
offers_df.show(10, truncate=False)

+--------------+--------+--------------------------------+---------+-------------+---+-----+------+------+
|discount_value|duration|id                              |min_value|offer_type   |web|email|mobile|social|
+--------------+--------+--------------------------------+---------+-------------+---+-----+------+------+
|10            |5.0     |4d5c57ea9a6940dd891ad53e9dbe8da0|10       |bogo         |1  |1    |1     |1     |
|10            |7.0     |ae264e3637204a6fb9bb56bc8210ddfd|10       |bogo         |0  |1    |1     |1     |
|0             |3.0     |5a8bc65990b245e5a138643cd4eb9837|0        |informational|0  |1    |1     |1     |
|5             |5.0     |f19421c1d4aa40978ebb69ca19b0e20d|5        |bogo         |1  |1    |1     |1     |
|2             |7.0     |2906b810c7d4411798c6938adc9daaa5|10       |discount     |1  |1    |1     |0     |
|3             |7.0     |2298d6c36e964ae4a3e7e9706d1fb8c2|7        |discount     |1  |1    |1     |1     |
|0             |4.0     |3f207df678b1

In [80]:
print(f"shape: {offers_df.count()}, {len(offers_df.columns)}")

shape: 10, 9


In [81]:
for col_name in offers_df.columns:
    print(f"{col_name}: {offers_df.select(col_name).distinct().count()} distinct values")

discount_value: 5 distinct values
duration: 5 distinct values
id: 10 distinct values
min_value: 5 distinct values
offer_type: 3 distinct values
web: 2 distinct values
email: 1 distinct values
mobile: 2 distinct values
social: 2 distinct values


In [82]:
offers_df.filter(col("offer_type") == "bogo").orderBy("discount_value").show()

+--------------+--------+--------------------+---------+----------+---+-----+------+------+
|discount_value|duration|                  id|min_value|offer_type|web|email|mobile|social|
+--------------+--------+--------------------+---------+----------+---+-----+------+------+
|             5|     5.0|f19421c1d4aa40978...|        5|      bogo|  1|    1|     1|     1|
|             5|     7.0|9b98b8c7a33c4b65b...|        5|      bogo|  1|    1|     1|     0|
|            10|     5.0|4d5c57ea9a6940dd8...|       10|      bogo|  1|    1|     1|     1|
|            10|     7.0|ae264e3637204a6fb...|       10|      bogo|  0|    1|     1|     1|
+--------------+--------+--------------------+---------+----------+---+-----+------+------+



In [83]:
offers_df.filter(col("offer_type") == "informational").orderBy("discount_value").show()

+--------------+--------+--------------------+---------+-------------+---+-----+------+------+
|discount_value|duration|                  id|min_value|   offer_type|web|email|mobile|social|
+--------------+--------+--------------------+---------+-------------+---+-----+------+------+
|             0|     3.0|5a8bc65990b245e5a...|        0|informational|  0|    1|     1|     1|
|             0|     4.0|3f207df678b143eea...|        0|informational|  1|    1|     1|     0|
+--------------+--------+--------------------+---------+-------------+---+-----+------+------+



In [84]:
offers_df.filter(col("offer_type") == "discount").orderBy("discount_value").show()

+--------------+--------+--------------------+---------+----------+---+-----+------+------+
|discount_value|duration|                  id|min_value|offer_type|web|email|mobile|social|
+--------------+--------+--------------------+---------+----------+---+-----+------+------+
|             2|     7.0|2906b810c7d441179...|       10|  discount|  1|    1|     1|     0|
|             2|    10.0|fafdcd668e3743c1b...|       10|  discount|  1|    1|     1|     1|
|             3|     7.0|2298d6c36e964ae4a...|        7|  discount|  1|    1|     1|     1|
|             5|    10.0|0b1e1539f2cc45b7b...|       20|  discount|  1|    1|     0|     0|
+--------------+--------+--------------------+---------+----------+---+-----+------+------+



In [85]:
spark.stop()